In [1]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.documents import Document

In [3]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [4]:
from langchain_community.vectorstores import Chroma

In [5]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)

# create vector store from the documents by embedding it using embedding model

In [6]:
vectorstore=Chroma.from_documents(
    documents=docs,
    embedding=hf_embeddings_api
)

In [7]:
base_retriever=vectorstore.as_retriever(
    search_type="similarity",#default search_type
    search_kwargs={
        "k":5
    }
)

In [ ]:

from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
    HuggingFaceEndpoint,
)

from dotenv import load_dotenv
import os

# hugging face free api endpoint is unreliable so trying locally
load_dotenv()
api_key = os.getenv("Hugging_face_api_token")

llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # repo_id="google/gemma-2-2b-it",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation", 
    huggingfacehub_api_token=api_key,
)
# os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"
# define the model
# llm = HuggingFacePipeline.from_model_id(
#     # this tinyllama is vert small for structured output tasks
#     # model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     model_id="google/gemma-2-2b-it",
#     task="text-generation",
#     pipeline_kwargs={
#         "max_new_tokens": 100,
#         "temperature": 0.5,
#     },
# )
model = ChatHuggingFace(llm=llm)

# Set up compressor using an LLM

In [9]:
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor


In [10]:
compression_retriever=ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=LLMChainExtractor.from_llm(model)
)

# Query the retriever

In [11]:
query="What is photosynthesis?"
compressed_results=compression_retriever.invoke(query)

In [12]:
for i,doc in enumerate(compressed_results):
    print(f"\n--Result {i+1} ---")
    print(doc.page_content)


--Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--Result 3 ---
Photosynthesis does not occur in animal cells.

--Result 4 ---
NO_OUTPUT. The context does not provide any information about photosynthesis.
